# S4 Binary Classification Demo

This notebook mirrors `s4_demo.py`: it builds the model, creates synthetic data, trains for 10 epochs, and prints accuracy.



In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from s4 import FFTConv


class S4BinaryClassifier(nn.Module):
    def __init__(self, input_dim: int, d_model: int, seq_len: int):
        super().__init__()
        self.seq_len = seq_len
        self.input_proj = nn.Linear(input_dim, d_model)
        self.s4 = FFTConv(
            d_model=d_model,
            l_max=seq_len,
            mode="s4",
            transposed=False,
            channels=1,
            dropout=0.0,
            d_state=64,
            rank=1,
        )
        self.readout = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1),
        )

    def forward(self, x):
        x = self.input_proj(x)
        y, _ = self.s4(x)
        y_last = y[:, -1, :]
        logits = self.readout(y_last).squeeze(-1)
        return logits


def build_dataset(num_samples: int = 512, seq_len: int = 8, input_dim: int = 3):
    torch.manual_seed(0)
    data = torch.randn(num_samples, seq_len, input_dim)
    labels = (data.sum(dim=(1, 2)) > 0).float()
    return TensorDataset(data, labels)



In [ ]:
def train(
    epochs: int = 10,
    batch_size: int = 32,
    seq_len: int = 8,
    input_dim: int = 3,
    d_model: int = 64,
    device: str = "cpu",
):
    dataset = build_dataset(num_samples=512, seq_len=seq_len, input_dim=input_dim)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = S4BinaryClassifier(input_dim=input_dim, d_model=d_model, seq_len=seq_len).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        correct = 0
        total = 0

        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * batch_x.size(0)
            preds = (logits.sigmoid() > 0.5).float()
            correct += (preds == batch_y).sum().item()
            total += batch_x.size(0)

        avg_loss = total_loss / total
        acc = correct / total * 100
        print(f"Epoch {epoch:02d} | loss={avg_loss:.4f} | acc={acc:.2f}%")

    return model


device = "cuda" if torch.cuda.is_available() else "cpu"
train(device=device)

